# Drug review effectiveness & side-effect classifier

**Dataset:** Drug Reviews (Druglib.com) — UCI Machine Learning Repository
Source: https://archive.ics.uci.edu/dataset/461/drug+review+dataset+druglib+com
Direct download: https://archive.ics.uci.edu/static/public/461/drug+review+dataset+druglib+com.zip

Citation: Kallumadi, S. & Graesser, F. (2018). Drug Reviews (Druglib.com) [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C55G6J

License: CC BY 4.0 — research/non-commercial use, attribution required.

**What this notebook does:**
1. Loads the real patient review dataset (4,143 rows, 8 columns, no missing values)
2. Cleans and vectorizes review text (TF-IDF)
3. Trains a classifier to predict `effectiveness` (5-step categorical) from review text
4. Trains a second classifier to predict `sideEffects` (5-step categorical) from review text
5. Evaluates both with accuracy, F1, and a confusion matrix
6. Aggregates per-drug predicted scores into a lookup table
7. Exports the trained models and the lookup table for Django to consume

**Setup before running:**
- Download the zip from the link above
- Extract `drugLibTrain_raw.tsv` and `drugLibTest_raw.tsv` into `./data/`


In [1]:
import pandas as pd
import numpy as np
import re
import joblib
import json
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

DATA_DIR = Path("../datasets")
OUTPUT_DIR = Path("../models")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42

## 1. Load the real dataset

In [2]:
train_df = pd.read_csv(DATA_DIR / "drugLibTrain_raw.tsv", sep="\t")
test_df = pd.read_csv(DATA_DIR / "drugLibTest_raw.tsv", sep="\t")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print()
print("Columns:", list(train_df.columns))
train_df.head(3)

Train shape: (3107, 9)
Test shape: (1036, 9)

Columns: ['Unnamed: 0', 'urlDrugName', 'rating', 'effectiveness', 'sideEffects', 'condition', 'benefitsReview', 'sideEffectsReview', 'commentsReview']


,Unnamed: 0,urlDrugName,rating,effectiveness,sideEffects,condition,benefitsReview,sideEffectsReview,commentsReview
0,2202,enalapril,4,Highly Effective,Mild Side Effects,management of congestive heart failure,slowed the progression of left ventricular dys...,"cough, hypotension , proteinuria, impotence , ...","monitor blood pressure , weight and asses for ..."
1,3117,ortho-tri-cyclen,1,Highly Effective,Severe Side Effects,birth prevention,Although this type of birth control has more c...,"Heavy Cycle, Cramps, Hot Flashes, Fatigue, Lon...","I Hate This Birth Control, I Would Not Suggest..."
2,1146,ponstel,10,Highly Effective,No Side Effects,menstrual cramps,I was used to having cramps so badly that they...,Heavier bleeding and clotting than normal.,I took 2 pills at the onset of my menstrual cr...


In [3]:
# Sanity check — confirm no missing values, as documented by UCI
print("Missing values per column (train):")
print(train_df.isnull().sum())
print()
print("Effectiveness categories:", train_df["effectiveness"].unique())
print("Side effects categories:", train_df["sideEffects"].unique())

Missing values per column (train):
Unnamed: 0            0
urlDrugName           0
rating                0
effectiveness         0
sideEffects           0
condition             1
benefitsReview       18
sideEffectsReview    75
commentsReview       12
dtype: int64

Effectiveness categories: <StringArray>
[      'Highly Effective',   'Marginally Effective',            'Ineffective',
 'Considerably Effective',   'Moderately Effective']
Length: 5, dtype: str
Side effects categories: <StringArray>
[            'Mild Side Effects',           'Severe Side Effects',
               'No Side Effects', 'Extremely Severe Side Effects',
         'Moderate Side Effects']
Length: 5, dtype: str


## 2. Clean review text

We combine the three free-text review fields (benefits, side effects, overall comment) into
a single text blob per row — this gives the model the most signal to work with.

In [4]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def combine_review_text(df):
    parts = [
        df["benefitsReview"].fillna(""),
        df["sideEffectsReview"].fillna(""),
        df["commentsReview"].fillna(""),
    ]
    combined = parts[0] + " " + parts[1] + " " + parts[2]
    return combined.apply(clean_text)

train_df["full_text"] = combine_review_text(train_df)
test_df["full_text"] = combine_review_text(test_df)

train_df = train_df[train_df["full_text"].str.len() > 0].reset_index(drop=True)
test_df = test_df[test_df["full_text"].str.len() > 0].reset_index(drop=True)

print("Train rows after cleaning:", len(train_df))
print("Test rows after cleaning:", len(test_df))
train_df[["urlDrugName", "condition", "full_text"]].head(2)

Train rows after cleaning: 3106
Test rows after cleaning: 1036


,urlDrugName,condition,full_text
0,enalapril,management of congestive heart failure,slowed the progression of left ventricular dys...
1,ortho-tri-cyclen,birth prevention,although this type of birth control has more c...


## 3. Train the effectiveness classifier

TF-IDF + Logistic Regression. This is intentionally a simple, interpretable pipeline —
easy to explain in a viva, fast to train, and a reasonable baseline for 5-class text
classification on ~3-4k rows. (A more complex model is not justified at this dataset size.)

In [5]:
X_train_text = train_df["full_text"]
y_train_eff = train_df["effectiveness"]
X_test_text = test_df["full_text"]
y_test_eff = test_df["effectiveness"]

effectiveness_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
])

effectiveness_pipeline.fit(X_train_text, y_train_eff)
pred_eff = effectiveness_pipeline.predict(X_test_text)

print("=== Effectiveness classifier ===")
print("Accuracy:", round(accuracy_score(y_test_eff, pred_eff), 4))
print("Macro F1:", round(f1_score(y_test_eff, pred_eff, average="macro"), 4))
print()
print(classification_report(y_test_eff, pred_eff, zero_division=0))

=== Effectiveness classifier ===
Accuracy: 0.4344
Macro F1: 0.38

                        precision    recall  f1-score   support

Considerably Effective       0.41      0.38      0.39       310
      Highly Effective       0.59      0.55      0.57       411
           Ineffective       0.34      0.52      0.41        82
  Marginally Effective       0.25      0.25      0.25        76
  Moderately Effective       0.27      0.28      0.27       157

              accuracy                           0.43      1036
             macro avg       0.37      0.40      0.38      1036
          weighted avg       0.44      0.43      0.44      1036



In [6]:
print("Confusion matrix (effectiveness) — rows = actual, columns = predicted")
labels_eff = sorted(y_test_eff.unique())
cm_eff = confusion_matrix(y_test_eff, pred_eff, labels=labels_eff)
cm_eff_df = pd.DataFrame(cm_eff, index=labels_eff, columns=labels_eff)
cm_eff_df

Confusion matrix (effectiveness) — rows = actual, columns = predicted


,Considerably Effective,Highly Effective,Ineffective,Marginally Effective,Moderately Effective
Considerably Effective,117,92,21,13,67
Highly Effective,109,227,31,15,29
Ineffective,7,12,43,12,8
Marginally Effective,15,12,14,19,16
Moderately Effective,36,42,18,17,44


## 4. Train the side-effect severity classifier (same approach, different target)

In [7]:
y_train_se = train_df["sideEffects"]
y_test_se = test_df["sideEffects"]

side_effect_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
])

side_effect_pipeline.fit(X_train_text, y_train_se)
pred_se = side_effect_pipeline.predict(X_test_text)

print("=== Side-effect severity classifier ===")
print("Accuracy:", round(accuracy_score(y_test_se, pred_se), 4))
print("Macro F1:", round(f1_score(y_test_se, pred_se, average="macro"), 4))
print()
print(classification_report(y_test_se, pred_se, zero_division=0))

=== Side-effect severity classifier ===
Accuracy: 0.4952
Macro F1: 0.4492

                               precision    recall  f1-score   support

Extremely Severe Side Effects       0.36      0.38      0.37        80
            Mild Side Effects       0.55      0.54      0.54       330
        Moderate Side Effects       0.39      0.35      0.37       236
              No Side Effects       0.67      0.69      0.68       268
          Severe Side Effects       0.26      0.31      0.28       122

                     accuracy                           0.50      1036
                    macro avg       0.45      0.45      0.45      1036
                 weighted avg       0.50      0.50      0.50      1036



## 5. Build the per-drug lookup table

The Flutter app doesn't call the model live per request — instead we run both classifiers
once across the *entire* dataset (train + test combined) and aggregate predictions per drug
name into a lookup table. Django serves from this table; it's fast and avoids needing a live
inference server for a student project demo.

We key on lowercase, stripped `urlDrugName` — Django will match this against
`Medicine.generic_name` (case-insensitive) at seed/sync time.

In [8]:
full_df = pd.concat([train_df, test_df], ignore_index=True)

full_df["pred_effectiveness"] = effectiveness_pipeline.predict(full_df["full_text"])
full_df["pred_side_effects"] = side_effect_pipeline.predict(full_df["full_text"])

# Map categorical labels to a numeric score so the app can sort/threshold easily
effectiveness_score_map = {
    "Ineffective": 1,
    "Marginally Effective": 2,
    "Moderately Effective": 3,
    "Considerably Effective": 4,
    "Highly Effective": 5,
}
side_effect_score_map = {
    "Severe Side Effects": 1,
    "Serious Side Effects": 2,
    "Moderate Side Effects": 3,
    "Mild Side Effects": 4,
    "No Side Effects": 5,
}

full_df["effectiveness_score"] = full_df["pred_effectiveness"].map(effectiveness_score_map)
full_df["side_effect_score"] = full_df["pred_side_effects"].map(side_effect_score_map)

full_df["drug_key"] = full_df["urlDrugName"].str.strip().str.lower()

lookup = (
    full_df.groupby("drug_key")
    .agg(
        review_count=("drug_key", "count"),
        avg_effectiveness_score=("effectiveness_score", "mean"),
        avg_side_effect_score=("side_effect_score", "mean"),
        most_common_condition=("condition", lambda s: s.mode().iloc[0] if not s.mode().empty else None),
    )
    .reset_index()
)

lookup["avg_effectiveness_score"] = lookup["avg_effectiveness_score"].round(2)
lookup["avg_side_effect_score"] = lookup["avg_side_effect_score"].round(2)

print("Unique drugs in lookup table:", len(lookup))
lookup.sort_values("review_count", ascending=False).head(10)

Unique drugs in lookup table: 541


,drug_key,review_count,avg_effectiveness_score,avg_side_effect_score,most_common_condition
261,lexapro,74,3.57,3.70,depression
356,paxil,58,3.45,3.07,depression
415,retin-a,55,3.53,3.56,acne
452,synthroid,53,3.83,4.49,hypothyroidism
532,zoloft,52,3.75,3.85,depression
400,prozac,51,3.76,3.49,depression
167,effexor,46,3.54,2.83,depression
168,effexor-xr,46,3.65,3.22,depression
4,accutane,44,4.32,3.02,acne
97,chantix,44,4.32,3.08,smoking cessation


## 6. Export for Django

Saves:
- `models/effectiveness_pipeline.joblib` — trained TF-IDF + LogisticRegression pipeline (kept in case you want live inference later)
- `models/side_effect_pipeline.joblib` — same, for side effects
- `models/drug_insight_lookup.json` — the precomputed per-drug table Django actually loads at runtime

`drug_insight_lookup.json` is the file your Django app needs. The two `.joblib` pipelines
are optional — keep them in the repo as evidence of the trained model, but the Django
integration in this plan reads only the JSON lookup, since on-demand TF-IDF inference
inside a request/response cycle is unnecessary overhead for a precomputed, static dataset.

In [9]:
joblib.dump(effectiveness_pipeline, OUTPUT_DIR / "effectiveness_pipeline.joblib")
joblib.dump(side_effect_pipeline, OUTPUT_DIR / "side_effect_pipeline.joblib")

lookup_records = lookup.to_dict(orient="records")
with open(OUTPUT_DIR / "drug_insight_lookup.json", "w") as f:
    json.dump(lookup_records, f, indent=2)

print("Saved:")
print(" -", OUTPUT_DIR / "effectiveness_pipeline.joblib")
print(" -", OUTPUT_DIR / "side_effect_pipeline.joblib")
print(" -", OUTPUT_DIR / "drug_insight_lookup.json")
print()
print("Sample lookup record:")
print(json.dumps(lookup_records[0], indent=2))

Saved:


 - ..\models\effectiveness_pipeline.joblib
 - ..\models\side_effect_pipeline.joblib
 - ..\models\drug_insight_lookup.json

Sample lookup record:
{
  "drug_key": "abilify",
  "review_count": 8,
  "avg_effectiveness_score": 3.75,
  "avg_side_effect_score": 3.57,
  "most_common_condition": "depression"
}


## Notes for the report

- This model is trained and evaluated entirely on the UCI Druglib.com dataset — real
  patient-submitted reviews, not synthetic or self-generated data.
- Drug names in this dataset are mostly US/Western brand and generic names. Matching against
  Nepal pharmacy catalogue entries is done on **generic name**, not brand name, since brand
  names differ by market (e.g. Paracetamol vs Tylenol). Coverage will be partial — not every
  medicine in the Nepal catalogue will have a matching review in this dataset, and the app
  should treat missing lookup entries as "no community data available" rather than erroring.
- Report both per-class metrics (above) and explain *why* macro F1 is the more honest metric
  here over accuracy — class imbalance exists in both targets (e.g. far more "Highly Effective"
  reviews than "Ineffective" ones), so accuracy alone would be a misleading.
